<a href="https://colab.research.google.com/github/dalkhaseeb-D/at-risk-student-prediction/blob/main/notebooks/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import zipfile
import requests
from pathlib import Path

DATA_DIR = Path("/content/oulad_project/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

url = "https://archive.ics.uci.edu/static/public/349/open%2Buniversity%2Blearning%2Banalytics%2Bdataset.zip"
zip_path = DATA_DIR / "oulad.zip"

response = requests.get(url)
response.raise_for_status()
zip_path.write_bytes(response.content)

with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall(DATA_DIR)

print("OULAD dataset downloaded and extracted successfully.")
print(os.listdir(DATA_DIR))

OULAD dataset downloaded and extracted successfully.
['oulad.zip', 'studentRegistration.csv', 'OULAD.names', 'vle.csv', 'studentVle.csv', 'studentAssessment.csv', 'assessments.csv', 'courses.csv', 'studentInfo.csv']


In [4]:
import pandas as pd

small_files = [
    "courses.csv",
    "assessments.csv",
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentAssessment.csv",
    "vle.csv"
]

tables = {}

for file in small_files:
    file_path = DATA_DIR / file
    tables[file] = pd.read_csv(file_path)
    print(f"{file}: {tables[file].shape[0]} rows, {tables[file].shape[1]} columns")

courses.csv: 22 rows, 3 columns
assessments.csv: 206 rows, 6 columns
studentInfo.csv: 32593 rows, 12 columns
studentRegistration.csv: 32593 rows, 5 columns
studentAssessment.csv: 173912 rows, 5 columns
vle.csv: 6364 rows, 6 columns


In [6]:
audit_results = []

for file, dataframe in tables.items():
    audit_results.append({
        "File": file,
        "Rows": dataframe.shape[0],
        "Columns": dataframe.shape[1],
        "Duplicate_Rows": dataframe.duplicated().sum(),
        "Missing_Values": dataframe.isna().sum().sum()
    })

audit_df = pd.DataFrame(audit_results)
display(audit_df)

print("\nStudent final-result distribution:")
display(tables["studentInfo.csv"]["final_result"].value_counts())

,File,Rows,Columns,Duplicate_Rows,Missing_Values
0,courses.csv,22,3,0,0
1,assessments.csv,206,6,0,0
2,studentInfo.csv,32593,12,0,0
3,studentRegistration.csv,32593,5,0,0
4,studentAssessment.csv,173912,5,0,0
5,vle.csv,6364,6,0,0



Student final-result distribution:


,count
final_result,
Pass,12361
Withdrawn,10156
Fail,7052
Distinction,3024


In [10]:
student_info = tables["studentInfo.csv"]
registration = tables["studentRegistration.csv"]

keys = ["code_module", "code_presentation", "id_student"]

student_data = student_info.merge(
    registration,
    on=keys,
    how="left"
)

cutoff_day = 28

eligible_students = student_data[
    student_data["date_unregistration"].isna() |
    (student_data["date_unregistration"] > cutoff_day)
].copy()

eligible_students["target_at_risk"] = eligible_students[
    "final_result"
].isin(["Fail", "Withdrawn"]).astype(int)

print("Original registrations:", len(student_data))
print("Eligible at day 28:", len(eligible_students))
print(
    eligible_students["target_at_risk"]
    .value_counts()
    .rename(index={0: "Successful", 1: "At Risk"})
)

TypeError: '>' not supported between instances of 'str' and 'int'

In [12]:
student_data["date_unregistration"] = pd.to_numeric(
    student_data["date_unregistration"],
    errors="coerce"
)

cutoff_day = 28

eligible_students = student_data[
    student_data["date_unregistration"].isna() |
    (student_data["date_unregistration"] > cutoff_day)
].copy()

eligible_students["target_at_risk"] = (
    eligible_students["final_result"]
    .isin(["Fail", "Withdrawn"])
    .astype(int)
)

print("Original registrations:", len(student_data))
print("Eligible at day 28:", len(eligible_students))
print("\nTarget distribution:")
print(
    eligible_students["target_at_risk"]
    .value_counts()
    .rename(index={0: "Successful", 1: "At Risk"})
)

Original registrations: 32593
Eligible at day 28: 27538

Target distribution:
target_at_risk
Successful    15385
At Risk       12153
Name: count, dtype: int64
